In [ ]:
import pandas as pd
import pgeocode

In [ ]:
data_label = "1q2018"
raw_data_file = "PUDF_base1_" + data_label + "_tab.txt"
thcic_data_file = "provider_zip_codes/Facility_type"+data_label+"_tab_ZIP_handcoded.csv"
cleaned_data_file = "PUDF_base1_"+data_label+"_cleaned.csv"

In [3]:
#Read in the base_1 tables
df = pd.read_csv(raw_data_file,sep = "\t", low_memory=False)

In [4]:
# df = df.loc[0:1000,:] # for testing only, remove this line when running on the full dataset

In [5]:
# Drop fields 23-31,83-166, 168 (leaving penultimate column 167)
df = df.drop(columns=df.columns[82:166])
df = df.drop(columns=df.columns[22:31])
df = df.drop(columns=df.columns[-1])

In [ ]:
# Combine SPEC_UNIT codes into one concatenated column; Drop the individual SPEC_UNIT columns
df['SPEC_UNIT'] = ''
for column in ['SPEC_UNIT_1', 'SPEC_UNIT_2', 'SPEC_UNIT_3', 'SPEC_UNIT_4', 'SPEC_UNIT_5']:
    df['SPEC_UNIT'] += df[column].fillna('')

df = df.drop(columns = ['SPEC_UNIT_1', 'SPEC_UNIT_2', 'SPEC_UNIT_3', 'SPEC_UNIT_4', 'SPEC_UNIT_5'])

In [8]:
# Intialize DIAG_CODES_OA as a list (diagnostic codes present on arrival) with the Admitting Diagnostic code
df['DIAG_CODES_OA'] = df['ADMITTING_DIAGNOSIS'].str[0:3].apply(lambda x: [x])
# Append to the list values in which the diagnosis is present on arrival "Y"
for i in range(18,68,2):
    mask = df.iloc[:,i+1] != "Y"
    df.loc[mask, df.columns[i]] = ""
    codes = df.iloc[:,i].str[0:3]
    df['DIAG_CODES_OA'] = [
        lst + ([code] if code else [])
        for lst, code in zip(df['DIAG_CODES_OA'], codes)
    ]

In [ ]:
# Drop columns for Diagnostic Codes
df = df.drop(columns = df.columns[17:68])

df.drop(columns = ['SPEC_UNIT'], inplace=True) # information from SPEC_UNIT is not available at point of admission
df.drop(columns = ['PAT_STATUS'], inplace=True) # information from PAT_STATUS is not available at point of admission

We convert the principal diagnostic codes into one of the 22 chapters as defined in https://ftp.cdc.gov/pub/health_statistics/nchs/publications/ICD10CM/2022/icd10cm-tabular-2022-April-1.pdf ,

    
    1. Certain infectious and parasitic diseases (A00-B99)
    2. Neoplasms (C00-D49),
    3. Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism (D50-D89),
    4. Endocrine, nutritional and metabolic diseases (E00-E89),
    5. Mental, Behavioral and Neurodevelopmental disorders (F01-F99),
    6. Diseases of the nervous system (G00-G99),
    7. Diseases of the eye and adnexa (H00-H59),
    8. Diseases of the ear and mastoid process (H60-H95),
    9. Diseases of the circulatory system (I00-I99),
    10. Diseases of the respiratory system (J00-J99),
    11. Diseases of the digestive system (K00-K95),
    12. Diseases of the skin and subcutaneous tissue (L00-L99),
    13. Diseases of the musculoskeletal system and connective tissue (M00-M99),
    14. Diseases of the genitourinary system (N00-N99),
    15. Pregnancy, childbirth and the puerperium (O00-O9A),
    16. Certain conditions originating in the perinatal period (P00-P96),
    17. Congenital malformations, deformations and chromosomal abnormalities (Q00-Q99),
    18. Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified (R00-R99),
    19. Injury, poisoning and certain other consequences of external causes (S00-T88),
    20. External causes of morbidity (V00-Y99),
    21. Factors influencing health status and contact with health services (Z00-Z99),
    22. Codes for special purposes (U00-U85)

In [10]:
# Create a dictionary with Chapter Number as the key, [lower bound of 3 digit codes, upper bound of 3 digit code, description] as the value
code_dict = {1 : ['A00','B99','Certain infectious and parasitic diseases'],
             2 : ['C00','D49','Neoplasms'],
             3 : ['D50','D89','Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism'],
             4 : ['E00','E89','Endocrine, nutritional and metabolic diseases' ],
             5 : ['F01','F99','Mental, Behavioral and Neurodevelopmental disorders' ],
             6 : ['G00','G99', 'Diseases of the nervous system'],
             7 : ['H00','H59','Diseases of the eye and adnexa'],
             8 : ['H60','H95','Diseases of the ear and mastoid process'],
             9 : ['I00','I99','Diseases of the circulatory system'],
             10 : ['J00','J99','Diseases of the respiratory system'],
             11 : ['K00','K95','Diseases of the digestive system'],
             12 : ['L00','L99','Diseases of the skin and subcutaneous tissue'],
             13 : ['M00','M99','Diseases of the musculoskeletal system and connective tissue'],
             14 : ['N00','N99','Diseases of the genitourinary system'],
             15 : ['O00','O9A','Pregnancy, childbirth and the puerperium'],
             16 : ['P00','P96','Certain conditions originating in the perinatal period'],
             17 : ['Q00','Q99','Congenital malformations, deformations and chromosomal abnormalities'],
             18 : ['R00','R99','Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified'],
             19 : ['S00','T88','Injury, poisoning and certain other consequences of external causes'],
             20 : ['V00','Y99','External causes of morbidity'],
             21 : ['Z00','Z99','Factors influencing health status and contact with health services'],
             22 : ['U00','U85','Codes for special purposes']}

# For each chapter, create a new column that runs through the DIAG_CODES_OA list and checks if there is any code between the lower and upper bound lexicographically
for key,value in code_dict.items():
    df['CODE_'+str(key)] = df['DIAG_CODES_OA'].apply(
    lambda codes: int(any(
        isinstance(code, str) and value[0] <= code <= value[1]
        for code in codes
    )))

# Note: Chapter 15 has codes of a different form (e.g. 'O9A'), but this still is captured in the lexicographic ordering

In [ ]:
#Dropping rows with missing or invalidvalues in any of the columns
df = df.dropna()

df=df.drop(df[df['PAT_ZIP'] == '`'].index)
df=df.drop(df[df['RACE'] == '`'].index)
df=df.drop(df[df['ETHNICITY'] == '`'].index)
df=df.drop(df[df['PAT_STATUS'] == '`'].index)
df=df.drop(df[df['PAT_AGE'] == '`'].index)
df=df.drop(df[df['PAT_COUNTRY'] == '`'].index)
df=df.drop(df[df['SOURCE_OF_ADMISSION'] == '`'].index)
df=df.drop(df[df['PAT_COUNTRY'] == '`'].index)

df=df.drop(df[df['SEX_CODE'] == 'U'].index)

df=df.drop(df[df['SOURCE_OF_ADMISSION'] == '9'].index) # These are "information not available" field values
df=df.drop(df[df['TYPE_OF_ADMISSION'] == '9'].index) # These are "information not available" field values
df=df.drop(df[df['SOURCE_OF_ADMISSION'].isin(['9','`'])].index) # These are "information not available" field values
df=df.drop(df[df['TYPE_OF_ADMISSION'].isin(['9','`'])].index) # These are "information not available" field values

#Changing data types of columns to int
df.PAT_ZIP = df.PAT_ZIP.astype(int)
df.RACE = df.RACE.astype(int)
df.ETHNICITY = df.ETHNICITY.astype(int)
df.PAT_STATUS = df.PAT_STATUS.astype(int)
df.PAT_AGE = df.PAT_AGE.astype(int)

#Changing data types of columns to string
df['SOURCE_OF_ADMISSION'] = df['SOURCE_OF_ADMISSION'].astype(str)
df['TYPE_OF_ADMISSION'] = df['TYPE_OF_ADMISSION'].astype(str)

Let's determine the rural vs. urban status of the patients by their zip codes

In [12]:
df=df.drop(df[df['PAT_ZIP']<1000].index) # drop patients with zip codes that have only 3 digits 

In [13]:
# !conda install -c conda-forge -n erdos_ds_environment pgeocode

In [ ]:
# 1. Initialize pgeocode for USA ZIP codes
nomi = pgeocode.Nominatim('us')

# 2. Define the set of Texas rural counties 
# Based on the official Texas Comptroller / Census definitions
texas_rural_counties = {
    'Anderson', 'Andrews', 'Aransas', 'Archer', 'Armstrong', 'Atascosa', 'Austin', 
    'Bailey', 'Bandera', 'Baylor', 'Bee', 'Blanco', 'Borden', 'Bosque', 'Brewster', 
    'Briscoe', 'Brooks', 'Brown', 'Burleson', 'Burnet', 'Caldwell', 'Calhoun', 
    'Callahan', 'Camp', 'Carson', 'Cass', 'Castro', 'Childress', 'Clay', 'Cochran', 
    'Coke', 'Coleman', 'Collingsworth', 'Colorado', 'Comanche', 'Concho', 'Cooke', 
    'Cottle', 'Crane', 'Crockett', 'Crosby', 'Culberson', 'Dallam', 'Dawson', 
    'Deaf Smith', 'Delta', 'DeWitt', 'Dickens', 'Dimmit', 'Donley', 'Duval', 
    'Eastland', 'Edwards', 'Erath', 'Falls', 'Fannin', 'Fayette', 'Fisher', 'Floyd', 
    'Foard', 'Franklin', 'Freestone', 'Frio', 'Gaines', 'Garza', 'Gillespie', 
    'Glasscock', 'Goliad', 'Gonzales', 'Gray', 'Grimes', 'Hale', 'Hall', 'Hamilton', 
    'Hansford', 'Hardeman', ' Hartley', 'Haskell', 'Hemphill', 'Hill', 'Hockley', 
    'Hopkins', 'Houston', 'Howard', 'Hudspeth', 'Hutchinson', 'Irion', 'Jack', 
    'Jackson', 'Jasper', 'Jeff Davis', 'Jim Hogg', 'Jim Wells', 'Jones', 'Karnes', 
    'Kendall', 'Kenedy', 'Kent', 'Kimble', 'King', 'Kinney', 'Kleberg', 'Knox', 
    'La Salle', 'Lamar', 'Lamb', 'Lampasas', 'Lavaca', 'Lee', 'Leon', 'Limestone', 
    'Lipscomb', 'Live Oak', 'Llano', 'Loving', 'Lynn', 'Madison', 'Marion', 
    'Martin', 'Mason', 'Matagorda', 'McCulloch', 'McMullen', 'Medina', 'Menard', 
    'Milam', 'Mills', 'Mitchell', 'Montague', 'Moore', 'Morris', 'Motley', 
    'Navarro', 'Newton', 'Nolan', 'Ochiltree', 'Oldham', 'Palo Pinto', 'Panola', 
    'Parmer', 'Pecos', 'Polk', 'Presidio', 'Rains', 'Reagan', 'Real', 'Red River', 
    'Reeves', 'Refugio', 'Roberts', 'Robertson', 'Runnels', 'Sabine', 'San Augustine', 
    'San Jacinto', 'San Saba', 'Schleicher', 'Scurry', 'Shackleford', 'Shelby', 
    'Sherman', 'Somervell', 'Starr', 'Stephens', 'Sterling', 'Stonewall', 'Sutton', 
    'Swisher', 'Terrell', 'Terry', 'Throckmorton', 'Titus', 'Trinity', 'Tyler', 
    'Upshur', 'Upton', 'Uvalde', 'Val Verde', 'Van Zandt', 'Victoria', 'Walker', 
    'Waller', 'Ward', 'Washington', 'Webb', 'Wharton', 'Wheeler', 'Wichita', 
    'Wilbarger', 'Willacy', 'Winkler', 'Wise', 'Wood', 'Yoakum', 'Young', 'Zapata', 'Zavala'
}

# 3. Create a function to check a ZIP code
def check_rural_urban(zip_code):
    # Fetch information for the ZIP code
    info = nomi.query_postal_code(str(zip_code))
    
    # Ensure the code is valid and located in Texas
    if pd.isna(info['county_name']) or info['state_code'] != 'TX':
        return "Invalid TX ZIP Code"
    
    county = info['county_name'].replace(" County", "")
    
    # Check if county is on the rural list
    if county in texas_rural_counties:
        return 1
    else:
        return 0
    
# df['PAT_RURAL'] = df['PAT_ZIP'].apply(check_rural_urban)


Map patient's zip code to rural/urban status of county

In [ ]:
zip_to_rural_dict = dict() # define dictionary that maps zip codes to rural/urban status (1 for rural, 0 for urban)

for zip_code in df['PAT_ZIP'].unique():
    zip_to_rural_dict[zip_code] = check_rural_urban(zip_code)

df['PAT_RURAL'] = df['PAT_ZIP'].map(zip_to_rural_dict)

print(f"The number of rural patients is: {df['PAT_RURAL'].sum()}")
print(f"The number of urban patients is: {((df['PAT_RURAL']+1)%2).sum()}")
print(f"The number of patients with missing rural/urban status is: {df['PAT_RURAL'].isna().sum()}")

The number of rural patients is: 100698
The number of urban patients is: 567414


Map provider's THCIC code to zip code

In [16]:
df_thcic = pd.read_csv(thcic_data_file)

thcic_to_zip_dict = dict()

for idx, row in df_thcic.iterrows():
    zip_code = row['ZIP']
    thcic_id = row['THCIC_ID']
    thcic_to_zip_dict[thcic_id] = zip_code

del df_thcic

df['PROVIDER_ZIP'] = df['THCIC_ID'].map(thcic_to_zip_dict)
df['PROVIDER_ZIP'] = df['PROVIDER_ZIP'].astype(int)

In [ ]:
zip_to_rural_dict = dict() # define dictionary that maps zip codes to rural/urban status (1 for rural, 0 for urban)

for zip_code in df['PROVIDER_ZIP'].unique():
    zip_to_rural_dict[zip_code] = check_rural_urban(zip_code)

df['PROVIDER_RURAL'] = df['PROVIDER_ZIP'].map(zip_to_rural_dict)

print(f"The number of patients in rural hospital is: {df['PROVIDER_RURAL'].sum()}")
print(f"The number of patients in urban hospital is: {((df['PROVIDER_RURAL']+1)%2).sum()}")
print(f"The number of patients with missing rural/urban hospital status is: {df['PROVIDER_RURAL'].isna().sum()}")

The number of patients in rural hospital is: 45960
The number of patients in urban hospital is: 622152
The number of patients with missing rural/urban hospital status is: 0
The number of rural patients is: 100698
The number of urban patients is: 567414
The number of patients with missing rural/urban status is: 0


In [20]:
#Calculates the distance between patient and provider zip codes; adds a new column called PAT2PROV_DISTANCE

dist = pgeocode.GeoDistance('us')

df['PAT_ZIP'] = df['PAT_ZIP'].astype(str)
df['PROVIDER_ZIP'] = df['PROVIDER_ZIP'].astype(str)

df['PAT2PROV_DISTANCE'] = dist.query_postal_code(
    df['PAT_ZIP'].values, 
    df['PROVIDER_ZIP'].values
)

df.drop(df[df['PAT2PROV_DISTANCE']==0].index,inplace=True)

df['PAT_ZIP'] = df['PAT_ZIP'].astype(int)
df['PROVIDER_ZIP'] = df['PROVIDER_ZIP'].astype(int)

In [21]:
df.to_csv(cleaned_data_file, index=False)

Getting rid of the rows with na or ' (invalid) results in around 85% of original data left.